# Oakland corridor activity — third feasibility pilot

This is a **small, targeted gate check** before a full 2019–2025 notebook. Run top to bottom in the same Cuebiq Snowflake environment as pilots 1 and 2. It reuses the persisted 11-corridor geohash9 cover from pilot 2 and the Fruitvale device-month table from pilot 1. It does not upload the cover or repeat the 15.5-minute month query.

It asks four questions:

1. When does PAPA/US coverage appear, and does any provider have US data on selected 2019 dates?
2. What do the US `BLOCK_GROUP_ID` values actually look like, and how many observed visitors have confident HOME assignments?
3. How long does a one-snapshot sample denominator scan take once the CBG check is corrected?
4. How long does a **production-style, one-week extraction across all 11 corridors** take, including stop deduplication, local dates, five overlapping periods, and repeat-device aggregation?

Source reads always constrain the appropriate partition date and country, plus provider when PAPA is the target. Historical any-provider probes intentionally omit provider. Only grouped data and geography codes are exported; device IDs stay in Snowflake. Each query prints elapsed time. A week query taking more than about 10 minutes is itself a negative runtime result: interrupt, set `RUN_WEEK_BENCHMARK=False`, and continue to the final checklist. Record Snowflake query-history bytes, partitions, and credits separately for the HOME, denominator, and week queries.

## 1. Connection and controls

In [1]:
import time
from pathlib import Path
import pandas as pd
from IPython.display import display

%load_ext cuebiqmagic.magics
%init_cuebiq_data
snow_engine = get_ipython().user_ns['instance']


schema : args.schema='paas_cda_pe_v3' and database: args.database='cuebiq_data' used
The connection to the Snowflake cluster has been established using database 'cuebiq_data' and schema 'paas_cda_pe_v3'.

You can now retrieve the connection using the following statement:
snow_engine = get_ipython().user_ns['instance']

The connection offers two methods


snow_engine.execute_statement("<SQL Statement>") # execute the SQL statement.


snow_engine.read_sql("<SQL Statement>") # execute the SQL statement and return a Pandas DataFrame.


snow_engine.write_dataframe(dataframe:pd.DataFrame, table_name:str,schema:str, index_in=False, if_exists_in="append",method_in="multi")


In [2]:
SOURCE_SCHEMA='CUEBIQ_DATA.PAAS_CDA_PE_V3'
STOP_TABLE=f'{SOURCE_SCHEMA}.STOP_BY_EVENT_DATE_UPLEVELLED'
HOME_TABLE=f'{SOURCE_SCHEMA}.DEVICE_RECURRING_AREA'
TMP_SCHEMA='DEDICATED.EDDIT_EAST_BAY_TMP'
PROVIDER_ID,COUNTRY_CODE='PAPA','US'
COVER_TABLE=f'{TMP_SCHEMA}.TMP_EB_P2_CORRIDOR_GEOHASH9'
PREVIOUS_DEVICE_MONTH=f'{TMP_SCHEMA}.TMP_EB_QUICK_DEVICE_MONTH_PERIOD'
PREVIOUS_RAW_MONTH=f'{TMP_SCHEMA}.TMP_EB_QUICK_CORRIDOR_STOPS_RAW'
P=f'{TMP_SCHEMA}.TMP_EB_P3_'
T={'HOME_OBS':P+'HOME_OBSERVED','CBG_SEED':P+'CBG_SEED',
   'STOPS':P+'STOP_ROWS','DEVICE_DAY':P+'DEVICE_DAY_PERIOD'}
FRUITVALE_ID='Oakland BID:8'
HOME_DATE=20240114
WEEK_START,WEEK_END=20240108,20240114
EVENT_START,EVENT_END=20240107,20240115  # one surrounding event-date partition on either side
MAX_RAW_STOP_MINUTES=480
CBG_CROSSWALK_PATH=None  # optional CSV: block_group_id, geography, vintage
RUN_HISTORICAL_PROVIDER_CHECK=True
RUN_HOME_CHECK=True
RUN_DENOMINATOR=True
RUN_WEEK_BENCHMARK=True
EXPORT_DIR=Path('./activity/pilot3')
# Jan/Jul checks for 2021–2023, plus late 2023. Pilot 2 already checked 2019, 2020, 2024, 2025 PAPA dates.
PAPA_INTERMEDIATE_DATES=[20210110,20210710,20220110,20220710,
                         20230110,20230710,20231210]
HISTORICAL_ANY_PROVIDER_DATES=[20190110,20190710,20191210,20200110]
print('Week:',WEEK_START,WEEK_END,'HOME snapshot:',HOME_DATE)

Week: 20240108 20240114 HOME snapshot: 20240114


## 2. Helpers and persisted input checks

In [3]:
timings=[]
results={}
def run(label,query,source=False):
    print('Start:',label,flush=True)
    start=time.monotonic()
    try:
        out=snow_engine.read_sql(query)
    except Exception:
        timings.append({'step':label,'seconds':round(time.monotonic()-start,1),
                        'source_scan':source,'status':'ERROR'})
        raise
    out.columns=[str(c).lower() for c in out.columns]
    seconds=round(time.monotonic()-start,1)
    timings.append({'step':label,'seconds':seconds,'source_scan':source,'status':'OK'})
    print(f'Done: {label} ({seconds:.1f} s)',flush=True)
    return out

cover_check=run('Check existing complete cover',f"""
SELECT COUNT(*) AS cover_rows,COUNT(DISTINCT corridor_id) AS corridors,
       COUNT(DISTINCT geohash9) AS distinct_cells
FROM {COVER_TABLE}""")
display(cover_check)
if int(cover_check.iloc[0].cover_rows)!=218259 or int(cover_check.iloc[0].corridors)!=11:
    raise ValueError('Pilot 2 complete cover is missing or incomplete. Reuse its uploaded Snowflake table.')
prior_check=run('Check existing Fruitvale visitor cohort',f"""
SELECT COUNT(DISTINCT cuebiq_id) AS observed_devices
FROM {PREVIOUS_DEVICE_MONTH}
WHERE corridor_id='{FRUITVALE_ID}' AND year_month=202401 AND time_period='all'""")
display(prior_check)
if int(prior_check.iloc[0].observed_devices)==0:
    raise ValueError('Pilot 1 Fruitvale device-month table is missing or empty.')
OBSERVED_DEVICES=int(prior_check.iloc[0].observed_devices)

Start: Check existing complete cover
Done: Check existing complete cover (1.6 s)


,cover_rows,corridors,distinct_cells
0,218259,11,218021


Start: Check existing Fruitvale visitor cohort
Done: Check existing Fruitvale visitor cohort (1.4 s)


,observed_devices
0,3537


## 3. Coverage and provider checks

`LIMIT 1` on exact date partitions avoids national row counts. The 2019/2020 any-provider check asks whether those partitions exist for **US regardless of PAPA**. If any 2019 stop partition exists, a corridor-matched query lists its contributing provider IDs. Different providers cannot be assumed to supply a comparable longitudinal panel.

In [4]:
availability=[]
for asset,table,col,tag in [('stops',STOP_TABLE,'event_date',''),
                            ('home',HOME_TABLE,'snapshot_event_date',"AND tag_type_code='HOME'")]:
    for day in PAPA_INTERMEDIATE_DATES:
        q=f"""SELECT {col} AS sample_date FROM {table}
        WHERE provider_id='{PROVIDER_ID}' AND country_code='{COUNTRY_CODE}'
          AND {col}={day} {tag} LIMIT 1"""
        found=len(run(f'{asset} PAPA {day}',q,source=True))>0
        availability.append({'asset':asset,'date':day,'provider_scope':'PAPA','found':int(found)})
    if RUN_HISTORICAL_PROVIDER_CHECK:
        for day in HISTORICAL_ANY_PROVIDER_DATES:
            q=f"""SELECT {col} AS sample_date FROM {table}
            WHERE country_code='{COUNTRY_CODE}' AND {col}={day} {tag} LIMIT 1"""
            found=len(run(f'{asset} any provider {day}',q,source=True))>0
            availability.append({'asset':asset,'date':day,'provider_scope':'ANY','found':int(found)})
results['availability_new']=pd.DataFrame(availability)
display(results['availability_new'])
if RUN_HISTORICAL_PROVIDER_CHECK:
    found_days=results['availability_new'].query(
        "asset=='stops' and provider_scope=='ANY' and date<20200101 and found==1")['date'].tolist()
    if found_days:
        day=int(found_days[0])
        results['historical_corridor_provider']=run('2019 corridor-matched providers',f"""
        SELECT s.provider_id,c.corridor_name,COUNT(*) AS matched_stop_slices
        FROM {STOP_TABLE} s JOIN {COVER_TABLE} c ON s.geohash_id=c.geohash9
        WHERE s.country_code='{COUNTRY_CODE}' AND s.event_date={day}
        GROUP BY 1,2 ORDER BY 1,2""",source=True)
        display(results['historical_corridor_provider'])
    else:
        print('No US 2019 stops on the tested dates from any provider; no corridor scan run.')

Start: stops PAPA 20210110
Done: stops PAPA 20210110 (5.0 s)
Start: stops PAPA 20210710
Done: stops PAPA 20210710 (3.6 s)
Start: stops PAPA 20220110
Done: stops PAPA 20220110 (3.9 s)
Start: stops PAPA 20220710
Done: stops PAPA 20220710 (2.8 s)
Start: stops PAPA 20230110
Done: stops PAPA 20230110 (4.1 s)
Start: stops PAPA 20230710
Done: stops PAPA 20230710 (4.4 s)
Start: stops PAPA 20231210
Done: stops PAPA 20231210 (4.2 s)
Start: stops any provider 20190110
Done: stops any provider 20190110 (1.2 s)
Start: stops any provider 20190710
Done: stops any provider 20190710 (1.9 s)
Start: stops any provider 20191210
Done: stops any provider 20191210 (4.1 s)
Start: stops any provider 20200110
Done: stops any provider 20200110 (1.6 s)
Start: home PAPA 20210110
Done: home PAPA 20210110 (3.5 s)
Start: home PAPA 20210710
Done: home PAPA 20210710 (4.0 s)
Start: home PAPA 20220110
Done: home PAPA 20220110 (3.4 s)
Start: home PAPA 20220710
Done: home PAPA 20220710 (3.2 s)
Start: home PAPA 20230110
Don

,asset,date,provider_scope,found
0,stops,20210110,PAPA,0
1,stops,20210710,PAPA,0
2,stops,20220110,PAPA,0
3,stops,20220710,PAPA,0
4,stops,20230110,PAPA,0
5,stops,20230710,PAPA,1
6,stops,20231210,PAPA,1
7,stops,20190110,ANY,1
8,stops,20190710,ANY,1
9,stops,20191210,ANY,1


Start: 2019 corridor-matched providers
Done: 2019 corridor-matched providers (18.4 s)


,provider_id,corridor_name,matched_stop_slices
0,LIMA,Chinatown,784
1,LIMA,Downtown,1147
2,LIMA,Fruitvale,296
3,LIMA,Jack London,419
4,LIMA,Koreatown/Northgate,272
5,LIMA,Lake Merritt,1027
6,LIMA,Lakeshore,110
7,LIMA,Laurel,125
8,LIMA,Montclair,101
9,LIMA,Rockridge,179


## 4. HOME value format and confident visitor coverage

One source snapshot partition is joined to the 3,537 observed Fruitvale visitors from pilot 1. The small resulting table remains in Snowflake. **No 12-digit assumption is made** when measuring HOME coverage. Report actual code lengths and representative block-group values (never device IDs); blank strings and missing values are separated. This corrects pilot 2's zero-CBG diagnostic.

In [5]:
if RUN_HOME_CHECK:
    run('Persist one-day observed HOME assignments',f"""
    CREATE OR REPLACE TABLE {T['HOME_OBS']} AS
    WITH observed AS (
      SELECT DISTINCT cuebiq_id FROM {PREVIOUS_DEVICE_MONTH}
      WHERE corridor_id='{FRUITVALE_ID}' AND year_month=202401 AND time_period='all'
    )
    SELECT h.cuebiq_id,h.block_group_id,h.geohash_id,h.confidence_level
    FROM {HOME_TABLE} h JOIN observed v ON h.cuebiq_id=v.cuebiq_id
    WHERE h.provider_id='{PROVIDER_ID}' AND h.country_code='{COUNTRY_CODE}'
      AND h.snapshot_event_date={HOME_DATE} AND h.tag_type_code='HOME'
    QUALIFY ROW_NUMBER() OVER (PARTITION BY h.cuebiq_id
      ORDER BY h.confidence_level DESC NULLS LAST,h.block_group_id)=1
    """,source=True)
    results['home_quality']=run('Observed HOME quality',f"""
    SELECT COUNT(*) AS devices_with_home,
      COUNT_IF(block_group_id IS NOT NULL AND TRIM(block_group_id)<>'') AS devices_with_nonblank_cbg,
      COUNT_IF(confidence_level>=0.60) AS devices_confident,
      COUNT_IF(confidence_level>=0.60 AND block_group_id IS NOT NULL
        AND TRIM(block_group_id)<>'') AS devices_confident_cbg,
      COUNT_IF(geohash_id IS NOT NULL) AS devices_with_geohash,
      COUNT(DISTINCT block_group_id) AS distinct_cbg_values
    FROM {T['HOME_OBS']}""")
    results['home_quality'].insert(0,'observed_devices',OBSERVED_DEVICES)
    display(results['home_quality'])
    results['home_lengths']=run('Actual CBG value lengths',f"""
    SELECT LENGTH(TRIM(block_group_id)) AS value_length,
           COUNT(*) AS devices,COUNT_IF(confidence_level>=0.60) AS confident_devices
    FROM {T['HOME_OBS']}
    WHERE block_group_id IS NOT NULL AND TRIM(block_group_id)<>''
    GROUP BY 1 ORDER BY 1""")
    display(results['home_lengths'])
    results['home_examples']=run('Frequent CBG values (no device IDs)',f"""
    SELECT block_group_id,COUNT(*) AS devices
    FROM {T['HOME_OBS']}
    WHERE block_group_id IS NOT NULL AND TRIM(block_group_id)<>''
    GROUP BY 1 ORDER BY devices DESC,block_group_id LIMIT 15""")
    display(results['home_examples'])
else:
    print('HOME check skipped; denominator cannot run in this notebook.')

Start: Persist one-day observed HOME assignments
Done: Persist one-day observed HOME assignments (82.8 s)
Start: Observed HOME quality
Done: Observed HOME quality (0.5 s)


,observed_devices,devices_with_home,devices_with_nonblank_cbg,devices_confident,devices_confident_cbg,devices_with_geohash,distinct_cbg_values
0,3537,2910,2910,2327,2327,0,1294


Start: Actual CBG value lengths
Done: Actual CBG value lengths (0.5 s)


,value_length,devices,confident_devices
0,18,2910,2327


Start: Frequent CBG values (no device IDs)
Done: Frequent CBG values (no device IDs) (0.4 s)


,block_group_id,devices
0,US.CA.001.407200.2,181
1,US.CA.001.407200.4,68
2,US.CA.001.406100.1,67
3,US.CA.001.406202.3,52
4,US.CA.001.407200.1,42
5,US.CA.001.407101.1,34
6,US.CA.001.407200.3,28
7,US.CA.001.406500.3,27
8,US.CA.001.407500.2,23
9,US.CA.001.407400.1,22


## 5. Optional CBG geography crosswalk and one-day denominator

A supplied crosswalk can test match rate for the six origin categories, but the pilot does not invent geography from a code prefix. The one-day denominator selects up to 25 **actual nonblank** CBG values among observed visitors and counts all confident HOME devices in those same CBGs on the same snapshot date. This tests scan cost and join semantics. It is not a full regional denominator or a 2019 sample index. If 2019 PAPA data are unavailable, the original 2019-reference normalization is blocked regardless of this query's speed.

In [6]:
if RUN_HOME_CHECK:
    observed_cbg=run('Observed CBG counts for optional crosswalk',f"""
    SELECT block_group_id,COUNT(*) AS visitor_devices
    FROM {T['HOME_OBS']}
    WHERE block_group_id IS NOT NULL AND TRIM(block_group_id)<>''
    GROUP BY 1""")
    if CBG_CROSSWALK_PATH is None:
        results['crosswalk_gate']=pd.DataFrame([{
          'status':'MISSING_CROSSWALK','observed_cbgs':len(observed_cbg),
          'matched_cbgs':0,'matched_visitor_devices':0,'vintage':'UNKNOWN'}])
    else:
        xwalk=pd.read_csv(CBG_CROSSWALK_PATH,dtype={'block_group_id':str,'geography':str,'vintage':str})
        expected={'block_group_id','geography','vintage'}
        if expected-set(xwalk): raise ValueError(f'Crosswalk missing {expected-set(xwalk)}')
        if xwalk.block_group_id.duplicated().any(): raise ValueError('Duplicate block_group_id in crosswalk')
        categories={'Oakland','San Francisco','Rest East Bay','Rest Bay Area',
                    'Rest California','Rest US'}
        if not set(xwalk.geography.dropna()).issubset(categories):
            raise ValueError('Crosswalk uses unexpected geography labels')
        matched=observed_cbg.merge(xwalk,on='block_group_id',how='left')
        results['crosswalk_gate']=pd.DataFrame([{
          'status':'CHECKED','observed_cbgs':len(matched),
          'matched_cbgs':int(matched.geography.notna().sum()),
          'matched_visitor_devices':int(matched.loc[matched.geography.notna(),'visitor_devices'].sum()),
          'vintage':','.join(sorted(xwalk.vintage.dropna().unique()))}])
    display(results['crosswalk_gate'])

    if RUN_DENOMINATOR and len(observed_cbg):
        run('Persist top 25 observed CBGs',f"""
        CREATE OR REPLACE TABLE {T['CBG_SEED']} AS
        SELECT block_group_id,COUNT(*) AS visitor_devices
        FROM {T['HOME_OBS']}
        WHERE block_group_id IS NOT NULL AND TRIM(block_group_id)<>''
        GROUP BY 1 ORDER BY visitor_devices DESC,block_group_id LIMIT 25""")
        results['sample_denominator']=run('One-snapshot 25-CBG denominator',f"""
        SELECT s.block_group_id,s.visitor_devices,
               COUNT(DISTINCT h.cuebiq_id) AS sample_devices
        FROM {T['CBG_SEED']} s LEFT JOIN {HOME_TABLE} h
          ON h.block_group_id=s.block_group_id
         AND h.provider_id='{PROVIDER_ID}' AND h.country_code='{COUNTRY_CODE}'
         AND h.snapshot_event_date={HOME_DATE} AND h.tag_type_code='HOME'
         AND h.confidence_level>=0.60
        GROUP BY s.block_group_id,s.visitor_devices
        ORDER BY s.visitor_devices DESC,s.block_group_id""",source=True)
        display(results['sample_denominator'].describe(include='all'))
    else:
        print('Denominator skipped: no observed nonblank CBGs, or RUN_DENOMINATOR=False.')

Start: Observed CBG counts for optional crosswalk
Done: Observed CBG counts for optional crosswalk (0.4 s)


,status,observed_cbgs,matched_cbgs,matched_visitor_devices,vintage
0,MISSING_CROSSWALK,1294,0,0,UNKNOWN


Start: Persist top 25 observed CBGs
Done: Persist top 25 observed CBGs (1.2 s)
Start: One-snapshot 25-CBG denominator
Done: One-snapshot 25-CBG denominator (112.0 s)


,block_group_id,visitor_devices,sample_devices
count,25,25.000000,25.00000
unique,25,NaN,NaN
top,US.CA.001.407200.2,NaN,NaN
freq,1,NaN,NaN
mean,NaN,31.360000,537.84000
std,NaN,34.985568,199.81735
min,NaN,12.000000,228.00000
25%,NaN,15.000000,428.00000
50%,NaN,19.000000,491.00000
75%,NaN,28.000000,637.00000


## 6. One-week all-corridor production-shape benchmark

The source scan reads nine explicit `event_date` partitions, allowing one partition on either side of the Jan 8–14 **local start-date** window. It joins the existing complete cover and deduplicates event-date slices by corridor/device/geohash/start/duration. It then creates one visit per corridor/device/local day in each applicable time period. The five periods overlap and should never be summed. This is a bounded runtime and correctness test; repeat-device proportions describe only this sampled week, not a half-year. The week query may be the slowest cell.

In [7]:
if RUN_WEEK_BENCHMARK:
    run('Deduplicate all-corridor week',f"""
    CREATE OR REPLACE TABLE {T['STOPS']} AS
    WITH matched AS (
      SELECT c.corridor_id,c.corridor_name,s.cuebiq_id,
        s.geohash_id AS geohash9,s.event_date,s.processing_date,
        s.stop_zoned_datetime,s.dwell_time_minutes,
        TO_TIMESTAMP_TZ(s.stop_zoned_datetime) AS stop_start_ts,
        ROW_NUMBER() OVER (PARTITION BY c.corridor_id,s.cuebiq_id,s.geohash_id,
          s.stop_zoned_datetime,s.dwell_time_minutes
          ORDER BY s.processing_date DESC,s.event_date ASC) AS rn
      FROM {STOP_TABLE} s JOIN {COVER_TABLE} c ON s.geohash_id=c.geohash9
      WHERE s.provider_id='{PROVIDER_ID}' AND s.country_code='{COUNTRY_CODE}'
        AND s.event_date BETWEEN {EVENT_START} AND {EVENT_END}
        AND s.stop_zoned_datetime IS NOT NULL
        AND s.dwell_time_minutes>0 AND s.dwell_time_minutes<={MAX_RAW_STOP_MINUTES}
    )
    SELECT corridor_id,corridor_name,cuebiq_id,geohash9,event_date,
      stop_start_ts,dwell_time_minutes,
      TO_NUMBER(TO_CHAR(stop_start_ts,'YYYYMMDD')) AS visit_date,
      DAYOFWEEKISO(stop_start_ts) AS dow_iso,HOUR(stop_start_ts) AS hour_of_day
    FROM matched WHERE rn=1
      AND TO_NUMBER(TO_CHAR(stop_start_ts,'YYYYMMDD'))
          BETWEEN {WEEK_START} AND {WEEK_END}
    """,source=True)
    results['week_qa']=run('Week boundaries and corridor QA',f"""
    SELECT COUNT(*) AS stop_rows,
      COUNT(DISTINCT corridor_id) AS corridors,
      COUNT_IF(event_date<{WEEK_START} OR event_date>{WEEK_END}) AS event_slices_outside_week,
      COUNT_IF(visit_date<{WEEK_START} OR visit_date>{WEEK_END}) AS invalid_local_dates
    FROM {T['STOPS']}""")
    display(results['week_qa'])
    if int(results['week_qa'].iloc[0].invalid_local_dates):
        raise AssertionError('Stop extraction includes local dates outside pilot week')
    run('Build one visit per corridor-device-day-period',f"""
    CREATE OR REPLACE TABLE {T['DEVICE_DAY']} AS
    WITH p AS (
      SELECT corridor_id,corridor_name,cuebiq_id,visit_date,'all' AS time_period
        FROM {T['STOPS']}
      UNION ALL SELECT corridor_id,corridor_name,cuebiq_id,visit_date,'weekdays'
        FROM {T['STOPS']} WHERE dow_iso BETWEEN 1 AND 5
      UNION ALL SELECT corridor_id,corridor_name,cuebiq_id,visit_date,'weekends'
        FROM {T['STOPS']} WHERE dow_iso IN (6,7)
      UNION ALL SELECT corridor_id,corridor_name,cuebiq_id,visit_date,'nine-five'
        FROM {T['STOPS']} WHERE dow_iso BETWEEN 1 AND 5 AND hour_of_day BETWEEN 9 AND 16
      UNION ALL SELECT corridor_id,corridor_name,cuebiq_id,visit_date,'evening'
        FROM {T['STOPS']} WHERE hour_of_day BETWEEN 17 AND 22
    )
    SELECT corridor_id,corridor_name,cuebiq_id,visit_date,time_period,
           COUNT(*) AS raw_stop_count,1 AS visit_count
    FROM p GROUP BY 1,2,3,4,5""")
    results['week_corridors']=run('Weekly metrics by corridor and time period',f"""
    SELECT corridor_id,corridor_name,time_period,
           SUM(raw_stop_count) AS raw_stop_count,
           SUM(visit_count) AS visit_count,
           COUNT(DISTINCT cuebiq_id) AS unique_devices
    FROM {T['DEVICE_DAY']} GROUP BY 1,2,3 ORDER BY 2,3""")
    display(results['week_corridors'])
    results['week_repeat']=run('Repeat devices within observed week',f"""
    WITH device_period AS (
      SELECT corridor_id,corridor_name,time_period,cuebiq_id,
        COUNT(*) AS visit_days
      FROM {T['DEVICE_DAY']} GROUP BY 1,2,3,4
    )
    SELECT corridor_id,corridor_name,time_period,
           COUNT(*) AS unique_devices,
           COUNT_IF(visit_days>=2) AS repeat_devices,
           COUNT_IF(visit_days>=2)/NULLIF(COUNT(*),0) AS repeat_prop_week_only
    FROM device_period GROUP BY 1,2,3 ORDER BY 2,3""")
    display(results['week_repeat'])
    results['grain_qa']=run('Device-day grain QA',f"""
    SELECT COUNT(*) AS duplicated_groups FROM (
      SELECT corridor_id,cuebiq_id,visit_date,time_period
      FROM {T['DEVICE_DAY']} GROUP BY 1,2,3,4 HAVING COUNT(*)>1
    )""")
    display(results['grain_qa'])
    if int(results['grain_qa'].iloc[0].duplicated_groups):
        raise AssertionError('Duplicate corridor-device-day-period records')
    try:
        results['fruitvale_comparison']=run('Compare with pilot 1 Fruitvale week',f"""
        SELECT p.raw_stop_count AS pilot1_raw_stops,p.visit_count AS pilot1_visits,
               p.unique_devices AS pilot1_devices,
               w.raw_stop_count AS pilot3_raw_stops,w.visit_count AS pilot3_visits,
               w.unique_devices AS pilot3_devices
        FROM {PREVIOUS_RAW_MONTH} p JOIN (
          SELECT corridor_id,SUM(raw_stop_count) AS raw_stop_count,
                 SUM(visit_count) AS visit_count,
                 COUNT(DISTINCT cuebiq_id) AS unique_devices
          FROM {T['DEVICE_DAY']}
          WHERE corridor_id='{FRUITVALE_ID}' AND time_period='all'
          GROUP BY corridor_id
        ) w ON p.corridor_id=w.corridor_id
        WHERE p.year_month=202401 AND p.time_period='all'""")
        display(results['fruitvale_comparison'])
        print('A difference can reflect the extra surrounding event-date partitions; inspect week_qa.')
    except Exception as exc:
        print('Prior raw comparison table unavailable:',type(exc).__name__)
else:
    print('Week benchmark skipped; production-shape runtime remains untested.')

Start: Deduplicate all-corridor week
Done: Deduplicate all-corridor week (284.9 s)
Start: Week boundaries and corridor QA
Done: Week boundaries and corridor QA (0.5 s)


,stop_rows,corridors,event_slices_outside_week,invalid_local_dates
0,44457,11,0,0


Start: Build one visit per corridor-device-day-period
Done: Build one visit per corridor-device-day-period (1.0 s)
Start: Weekly metrics by corridor and time period
Done: Weekly metrics by corridor and time period (0.5 s)


,corridor_id,corridor_name,time_period,raw_stop_count,visit_count,unique_devices
0,Oakland BID:2,Chinatown,all,5050,3910,3218
1,Oakland BID:2,Chinatown,evening,1087,952,872
2,Oakland BID:2,Chinatown,nine-five,1941,1620,1365
3,Oakland BID:2,Chinatown,weekdays,3435,2700,2255
4,Oakland BID:2,Chinatown,weekends,1615,1210,1135
5,Oakland BID:1,Downtown,all,8418,6089,5003
6,Oakland BID:1,Downtown,evening,1738,1446,1338
7,Oakland BID:1,Downtown,nine-five,2902,2331,2025
8,Oakland BID:1,Downtown,weekdays,5699,4235,3614
9,Oakland BID:1,Downtown,weekends,2719,1854,1704


Start: Repeat devices within observed week
Done: Repeat devices within observed week (0.5 s)


...

Start: Device-day grain QA
Done: Device-day grain QA (0.5 s)


,duplicated_groups
0,0


Start: Compare with pilot 1 Fruitvale week
Done: Compare with pilot 1 Fruitvale week (0.6 s)


,pilot1_raw_stops,pilot1_visits,pilot1_devices,pilot3_raw_stops,pilot3_visits,pilot3_devices
0,5225,4121,3537,5225,4121,3537


A difference can reflect the extra surrounding event-date partitions; inspect week_qa.


## 7. Export grouped evidence and remaining gates

The CSVs contain grouped values only. Do not describe a week repeat rate as half-year loyalty. The 2019 baseline is viable only if a comparable provider has both historical corridor stops and HOME origins. A complete CBG geography crosswalk and a representative regional denominator benchmark are still necessary before normalized origin outputs. For full extraction, process bounded date chunks and reconcile local-date boundaries while scanning all corridors together.

In [8]:
results['timings']=pd.DataFrame(timings)
if not results['availability_new'].empty:
    coverage=results['availability_new'].query("provider_scope=='PAPA'")
    found_by_asset={a:(int(g.loc[g['found'].eq(1),'date'].min())
                        if g['found'].eq(1).any() else None)
                    for a,g in coverage.groupby('asset')}
else:
    found_by_asset={}
status=pd.DataFrame([
 {'gate':'PAPA stop 2021–2023 sampled coverage','result':
  str(found_by_asset.get('stops')) if found_by_asset.get('stops') else 'NONE_ON_TESTED_DATES'},
 {'gate':'PAPA HOME 2021–2023 sampled coverage','result':
  str(found_by_asset.get('home')) if found_by_asset.get('home') else 'NONE_ON_TESTED_DATES'},
 {'gate':'US 2019 any-provider sampled coverage','result':
  'FOUND' if RUN_HISTORICAL_PROVIDER_CHECK and
   results['availability_new'].query("provider_scope=='ANY' and date<20200101 and found==1").shape[0]
   else 'NONE_ON_TESTED_DATES_OR_NOT_RUN'},
 {'gate':'Observed HOME CBG format and confidence','result':'MEASURED' if 'home_quality' in results else 'NOT_RUN'},
 {'gate':'CBG geographic crosswalk','result':
  results['crosswalk_gate'].iloc[0]['status'] if 'crosswalk_gate' in results else 'NOT_RUN'},
 {'gate':'25-CBG same-date sample cost','result':'MEASURED' if 'sample_denominator' in results else 'NOT_RUN'},
 {'gate':'All-corridor production-shape week cost','result':'MEASURED' if 'week_corridors' in results else 'NOT_RUN'},
 {'gate':'2019-reference normalization','result':'BLOCKED_UNTIL_COMPARABLE_2019_COVERAGE'},
])
results['decision_checklist']=status
EXPORT_DIR.mkdir(parents=True,exist_ok=True)
for name,frame in results.items():
    frame.to_csv(EXPORT_DIR/f'{name}.csv',index=False)
display(results['timings'])
display(status)
print('Grouped diagnostics:',EXPORT_DIR.resolve())

,step,seconds,source_scan,status
0,Check existing complete cover,1.6,False,OK
1,Check existing Fruitvale visitor cohort,1.4,False,OK
2,stops PAPA 20210110,5.0,True,OK
3,stops PAPA 20210710,3.6,True,OK
4,stops PAPA 20220110,3.9,True,OK
5,stops PAPA 20220710,2.8,True,OK
6,stops PAPA 20230110,4.1,True,OK
7,stops PAPA 20230710,4.4,True,OK
8,stops PAPA 20231210,4.2,True,OK
9,stops any provider 20190110,1.2,True,OK


,gate,result
0,PAPA stop 2021–2023 sampled coverage,20230710
1,PAPA HOME 2021–2023 sampled coverage,20230710
2,US 2019 any-provider sampled coverage,FOUND
3,Observed HOME CBG format and confidence,MEASURED
4,CBG geographic crosswalk,MISSING_CROSSWALK
5,25-CBG same-date sample cost,MEASURED
6,All-corridor production-shape week cost,MEASURED
7,2019-reference normalization,BLOCKED_UNTIL_COMPARABLE_2019_COVERAGE


Grouped diagnostics: /home/jovyan/eddit-east-bay/activity/pilot3
